# Processing raw velocity data

Computes velocity from `treadmillPosition` in the intermediate behavior JSON
(output of `process_json_behavior_data.py`) and saves it as
`[[time_s, velocity], ...]` pairs to `filtered_velocity.json`.

**Pipeline position:**
```
raw .tdml
  → process_json_behavior_data.py  → <session>.json   (treadmillPosition pairs)
  → this notebook                  → filtered_velocity.json  ([[time_s, vel], ...])
  → BehaviorData.resample_to_imaging()               (uniform imaging-rate grid)
```

**What is saved:** raw event-driven velocity with per-event timestamps.  
Gaussian smoothing is shown below for inspection only and is **not** written
to disk — apply it after resampling to the imaging grid in the analysis workflow.

**"filtered"** in the filename now refers to optional outlier removal (NaN-drop),
not Gaussian smoothing.

In [ ]:
import json
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter

## Configuration

Set `behavior_json` to the `.json` file produced by `process_json_behavior_data.py`  
and `output_dir` to the `.sima/behavior/` folder where `filtered_velocity.json` will be saved.

In [ ]:
# Path to the intermediate JSON from process_json_behavior_data.py
behavior_json = "/data2/gergely/BehaviorData/<mouse_id>/<session>.json"

# Path to the .sima/behavior/ folder where filtered_velocity.json will be written
output_dir = "/data2/gergely/invivo_DATA/sleep/<mouse_id>/TSeries-<...>.sima/behavior"

## Load and compute velocity

Velocity at each BehaviorMate event = Δnormalized_position / Δtime_s.  
The timestamp for each velocity sample is the start of the inter-event interval.

In [ ]:
with open(behavior_json) as f:
    data = json.load(f)

tp = np.array(data["treadmillPosition"])   # shape (N, 2): [time_s, norm_position]
dt = np.diff(tp[:, 0])                     # inter-event durations
dp = np.diff(tp[:, 1])                     # position changes (normalized)
velocity = dp / dt                          # shape (N-1,)
times = tp[:-1, 0]                          # timestamp = start of interval

velocity_ts = np.column_stack([times, velocity])   # shape (N-1, 2)

print(f"events:           {len(velocity_ts)}")
print(f"time range:       {times[0]:.2f} s → {times[-1]:.2f} s")
print(f"recordingDuration:{data['recordingDuration']} s")
print(f"velocity range:   {velocity.min():.4f} → {velocity.max():.4f} (normalized pos / s)")

## Inspect velocity

The lower panel shows a Gaussian-smoothed preview (σ=2 samples) for visual
inspection.  This smoothing is **not** saved — the raw event-driven signal is
what gets written to disk and later resampled to the imaging grid.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(times, velocity, lw=0.4, alpha=0.7, color="C0")
axes[0].set_ylabel("Velocity (norm. pos / s)")
axes[0].set_title("Raw event-driven velocity")

smoothed = gaussian_filter(velocity.astype(float), sigma=2)
axes[1].plot(times, smoothed, lw=0.8, color="C1")
axes[1].set_ylabel("Velocity (σ=2 preview)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Gaussian preview only — not saved")

plt.tight_layout()
plt.show()

## Outlier removal (optional)

If the plot above shows spikes caused by encoder glitches, mark the
corresponding event-index ranges in `outlier_slices` below.  Those events
are dropped from the time series before saving; `resample_to_imaging()` will
bridge the gap via linear interpolation.

Leave `outlier_slices = []` to skip this step.

In [ ]:
# Add (start, stop) index pairs for events to remove, e.g. [(13400, 13409)].
outlier_slices = [
    # (start, stop),
]

if outlier_slices:
    mask = np.ones(len(velocity), dtype=bool)
    for start, stop in outlier_slices:
        mask[start:stop] = False
    velocity_ts = velocity_ts[mask]
    print(f"Dropped {(~mask).sum()} events; {len(velocity_ts)} remaining.")
else:
    print("No outlier slices defined — all events retained.")

## Save

Writes `[[time_s, velocity], ...]` pairs — the format expected by
`BehaviorData.resample_to_imaging()`.

In [ ]:
output_path = join(output_dir, "filtered_velocity.json")
with open(output_path, "w") as f:
    json.dump(velocity_ts.tolist(), f, indent=4)
print(f"Saved {len(velocity_ts)} events → {output_path}")

## Verify

In [ ]:
with open(output_path) as f:
    loaded = np.array(json.load(f))

assert loaded.ndim == 2 and loaded.shape[1] == 2, f"unexpected shape: {loaded.shape}"
assert np.all(np.diff(loaded[:, 0]) > 0), "timestamps are not strictly increasing"

print(f"shape:      {loaded.shape}")
print(f"time range: {loaded[0, 0]:.3f} s → {loaded[-1, 0]:.3f} s")
print("OK")